# Cat and Dog Image Classification

In [ ]:
from pathlib import Path
import timm
import os
import copy
import zipfile
import random

import numpy as np
import matplotlib.pyplot as plt

import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader
import torchvision.transforms as transforms
from torchvision.datasets import ImageFolder
import optuna

In [ ]:
SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
torch.backends.cudnn.deterministic = True

if torch.cuda.is_available():
    DEVICE = torch.device("cuda")
elif torch.backends.mps.is_available():
    DEVICE = torch.device("mps")
else:
    DEVICE = torch.device("cpu")
print(f"Using device: {DEVICE}")

In [ ]:
PROJECT_DIR = Path.cwd()
DATA_DIR = PROJECT_DIR / "catdog_data"

OUTPUT_DIR = PROJECT_DIR / "outputs"
OUTPUT_DIR.mkdir(exist_ok=True)

# Keep this in case the dataset is stored as a zip in future
ZIP_PATH = PROJECT_DIR / "catdog_data.zip"
if not DATA_DIR.exists() and ZIP_PATH.exists():
    with zipfile.ZipFile(ZIP_PATH, "r") as z:
        z.extractall(PROJECT_DIR)
    print("Dataset extracted.")

if not DATA_DIR.is_dir():
    raise FileNotFoundError(f"Dataset not found: {DATA_DIR}")

# Hyperparameters
IMG_SIZE = 128
BATCH_SIZE = 32

# macOS / VS Code notebook setting
NUM_WORKERS = 0
PIN_MEMORY = DEVICE.type == "cuda"

# ImageNet statistics
MEAN = [0.485, 0.456, 0.406]
STD = [0.229, 0.224, 0.225]

# Local output files
MODEL_PATH = OUTPUT_DIR / "catdog_model.pth"
VIZ_DIR = OUTPUT_DIR / "visualizations"
VIZ_DIR.mkdir(exist_ok=True)

# Optuna configuration
DEFAULT_HYPERPARAMS = {
    "BATCH_SIZE": 32,
    "NUM_EPOCHS": 30,
    "LR": 1e-3,
    "WEIGHT_DECAY": 1e-4,
    "PATIENCE": 5,
}

N_TRIALS = 20
RUN_OPTUNA_TUNING = True
RUN_BONUS = True

print(f"Project folder: {PROJECT_DIR}")
print(f"Dataset folder: {DATA_DIR}")
print(f"Using device: {DEVICE}")

## 1. Data loading and augmentation

In [ ]:
# Data Loaders (Task 1)

train_transforms = transforms.Compose([
    transforms.RandomResizedCrop(IMG_SIZE, scale=(0.8, 1.0)),
    transforms.RandomHorizontalFlip(p=0.5),
    transforms.RandomRotation(degrees=15),
    transforms.ColorJitter(brightness=0.3, contrast=0.3, saturation=0.2, hue=0.05),
    transforms.ToTensor(),
    transforms.Normalize(mean=MEAN, std=STD),
])

val_test_transforms = transforms.Compose([
    transforms.Resize(IMG_SIZE + 16),
    transforms.CenterCrop(IMG_SIZE),
    transforms.ToTensor(),
    transforms.Normalize(mean=MEAN, std=STD),
])

def get_dataloaders(data_dir, batch_size=BATCH_SIZE):
    train_set = ImageFolder(os.path.join(data_dir, "train"),      transform=train_transforms)
    val_set   = ImageFolder(os.path.join(data_dir, "validation"), transform=val_test_transforms)
    test_set  = ImageFolder(os.path.join(data_dir, "test"),       transform=val_test_transforms)

    train_loader = DataLoader(train_set, batch_size=batch_size, shuffle=True,
                              num_workers=NUM_WORKERS, pin_memory=PIN_MEMORY)
    val_loader   = DataLoader(val_set,   batch_size=batch_size, shuffle=False,
                              num_workers=NUM_WORKERS, pin_memory=PIN_MEMORY)
    test_loader  = DataLoader(test_set,  batch_size=batch_size, shuffle=False,
                              num_workers=NUM_WORKERS, pin_memory=PIN_MEMORY)

    print(f"Train: {len(train_set)} | Val: {len(val_set)} | Test: {len(test_set)}")
    print(f"Classes: {train_set.classes}")
    return train_set, val_set, test_set, train_loader, val_loader, test_loader

## 2. Custom CNN architecture

In [ ]:
# Model (Task 2)

class DoubleConvBlock(nn.Module):
    def __init__(self, in_ch, out_ch):
        super().__init__()
        self.block = nn.Sequential(
            nn.Conv2d(in_ch,  out_ch, kernel_size=3, padding=1),
            nn.BatchNorm2d(out_ch),
            nn.ReLU(inplace=True),
            nn.Conv2d(out_ch, out_ch, kernel_size=3, padding=1),
            nn.BatchNorm2d(out_ch),
            nn.ReLU(inplace=True),
            nn.MaxPool2d(2, 2),
        )

    def forward(self, x):
        return self.block(x)

class CatDogCNN(nn.Module):
    def __init__(self, num_classes=2):
        super().__init__()

        self.features = nn.Sequential(
            DoubleConvBlock(3,   32),   # 128 × 128 → 64 × 64
            DoubleConvBlock(32,  64),   # 64  × 64  → 32 × 32
            DoubleConvBlock(64,  128),  # 32  × 32  → 16 × 16
            DoubleConvBlock(128, 256),  # 16  × 16  →  8 ×  8
        )

        # Global Average Pooling: collapses (256, 8, 8) → (256,)
        self.gap = nn.AdaptiveAvgPool2d((1, 1))

        self.classifier = nn.Sequential(
            nn.Flatten(),
            nn.Linear(256, 128),
            nn.ReLU(inplace=True),
            nn.Dropout(p=0.5),
            nn.Linear(128, num_classes),
        )

    def forward(self, x):
        x = self.features(x)
        x = self.gap(x)
        x = self.classifier(x)
        return x

def count_params(model):
    return sum(p.numel() for p in model.parameters() if p.requires_grad)

## 3. Training and evaluation functions

In [ ]:
# Training

def train_one_epoch(model, loader, criterion, optimizer):
    model.train()
    running_loss, correct, total = 0.0, 0, 0
    for imgs, labels in loader:
        imgs, labels = imgs.to(DEVICE), labels.to(DEVICE)
        optimizer.zero_grad()
        outputs = model(imgs)
        loss = criterion(outputs, labels)
        loss.backward()
        optimizer.step()
        running_loss += loss.item() * imgs.size(0)
        _, preds = outputs.max(1)
        correct += preds.eq(labels).sum().item()
        total   += labels.size(0)
    return running_loss / total, correct / total

@torch.no_grad()
def evaluate(model, loader, criterion):
    model.eval()
    running_loss, correct, total = 0.0, 0, 0
    all_preds, all_labels = [], []
    for imgs, labels in loader:
        imgs, labels = imgs.to(DEVICE), labels.to(DEVICE)
        outputs = model(imgs)
        loss = criterion(outputs, labels)
        running_loss += loss.item() * imgs.size(0)
        _, preds = outputs.max(1)
        correct += preds.eq(labels).sum().item()
        total   += labels.size(0)
        all_preds.extend(preds.cpu().numpy())
        all_labels.extend(labels.cpu().numpy())
    return running_loss / total, correct / total, all_preds, all_labels

def train_model_for_optuna(model, train_loader, val_loader, trial, params):
    criterion = nn.CrossEntropyLoss()
    optimizer = optim.AdamW(model.parameters(), lr=params["LR"], weight_decay=params["WEIGHT_DECAY"])
    # ReduceLROnPlateau halves LR when val_loss stalls
    scheduler = optim.lr_scheduler.ReduceLROnPlateau(optimizer, mode="min",
                                                      factor=0.5, patience=params["PATIENCE"])

    best_val_loss   = float("inf")

    for epoch in range(1, params["NUM_EPOCHS"] + 1):
        tr_loss, tr_acc = train_one_epoch(model, train_loader, criterion, optimizer)
        vl_loss, vl_acc, _, _ = evaluate(model, val_loader, criterion)
        scheduler.step(vl_loss)

        if vl_loss < best_val_loss:
            best_val_loss  = vl_loss

        # Report intermediate objective value to Optuna
        trial.report(vl_loss, epoch)

        # Handle pruning based on the intermediate value
        if trial.should_prune():
            raise optuna.exceptions.TrialPruned()

        print(
            f"Epoch [{epoch:02d}/{params['NUM_EPOCHS']}] "
            f"Train Loss: {tr_loss:.4f} | Train Acc: {tr_acc*100:.2f}% | "
            f"Val Loss: {vl_loss:.4f} | Val Acc: {vl_acc*100:.2f}%"
        )

    return best_val_loss

def run_training(model, train_loader, val_loader, hyperparameters=None):
    hparams = hyperparameters if hyperparameters is not None else DEFAULT_HYPERPARAMS

    criterion = nn.CrossEntropyLoss()
    optimizer = optim.AdamW(model.parameters(), lr=hparams["LR"], weight_decay=hparams["WEIGHT_DECAY"])
    scheduler = optim.lr_scheduler.ReduceLROnPlateau(optimizer, mode="min",
                                                      factor=0.5, patience=hparams["PATIENCE"])

    history = {"train_loss": [], "train_acc": [], "val_loss": [], "val_acc": []}
    best_val_loss   = float("inf")
    best_model_wts  = copy.deepcopy(model.state_dict())
    patience_counter = 0

    for epoch in range(1, hparams["NUM_EPOCHS"] + 1):
        tr_loss, tr_acc = train_one_epoch(model, train_loader, criterion, optimizer)
        vl_loss, vl_acc, _, _ = evaluate(model, val_loader, criterion)
        scheduler.step(vl_loss)

        history["train_loss"].append(tr_loss)
        history["train_acc"].append(tr_acc)
        history["val_loss"].append(vl_loss)
        history["val_acc"].append(vl_acc)

        improved = vl_loss < best_val_loss
        if improved:
            best_val_loss  = vl_loss
            best_model_wts = copy.deepcopy(model.state_dict())
            torch.save(model.state_dict(), MODEL_PATH)
            patience_counter = 0
            flag = " \u2190 best"
        else:
            patience_counter += 1
            flag = f" (no improvement {patience_counter}/{hparams['PATIENCE']})"


        print(
            f"Epoch [{epoch:02d}/{hparams['NUM_EPOCHS']}] "
            f"Train Loss: {tr_loss:.4f} | Train Acc: {tr_acc*100:.2f}% | "
            f"Val Loss: {vl_loss:.4f} | Val Acc: {vl_acc*100:.2f}%{flag}"
        )

        if patience_counter >= hparams["PATIENCE"]:
            print(f"\nEarly stopping triggered at epoch {epoch}.")
            break

    model.load_state_dict(best_model_wts)
    return history

## 4. Model visualisation

In [ ]:
# Visualisations (Task 3)

def plot_training_curves(history):
    """3a — Loss and accuracy curves."""
    epochs = range(1, len(history["train_loss"]) + 1)
    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))

    ax1.plot(epochs, history["train_loss"], label="Train Loss",  color="#E63946", linewidth=2)
    ax1.plot(epochs, history["val_loss"],   label="Val Loss",    color="#457B9D", linewidth=2, linestyle="--")
    ax1.set_title("Loss over Epochs", fontsize=14, fontweight="bold")
    ax1.set_xlabel("Epoch"); ax1.set_ylabel("Loss")
    ax1.legend(); ax1.grid(alpha=0.3)

    ax2.plot(epochs, [a * 100 for a in history["train_acc"]], label="Train Acc", color="#E63946", linewidth=2)
    ax2.plot(epochs, [a * 100 for a in history["val_acc"]],   label="Val Acc",  color="#457B9D", linewidth=2, linestyle="--")
    ax2.set_title("Accuracy over Epochs", fontsize=14, fontweight="bold")
    ax2.set_xlabel("Epoch"); ax2.set_ylabel("Accuracy (%)")
    ax2.legend(); ax2.grid(alpha=0.3)

    plt.tight_layout()
    path = os.path.join(VIZ_DIR, "training_curves.png")
    plt.savefig(path, dpi=150, bbox_inches="tight")
    plt.show()
    print(f"Saved: {path}")

def visualize_filters(model):
    """3b — Learned first-layer filters."""
    # Access first conv inside first DoubleConvBlock
    filters = model.features[0].block[0].weight.data.clone().cpu()
    n = min(32, filters.shape[0])
    fig, axes = plt.subplots(4, 8, figsize=(16, 8))
    for i, ax in enumerate(axes.flat):
        if i < n:
            f = filters[i]
            f = (f - f.min()) / (f.max() - f.min() + 1e-8)
            ax.imshow(f.permute(1, 2, 0).numpy())
        ax.axis("off")
    fig.suptitle("Learned First-Layer Filters", fontsize=16, fontweight="bold", y=1.01)
    plt.tight_layout()
    path = os.path.join(VIZ_DIR, "first_layer_filters.png")
    plt.savefig(path, dpi=150, bbox_inches="tight")
    plt.show()
    print(f"Saved: {path}")

def visualize_feature_maps(model, dataset, n_maps=8):
    """3c — Feature maps from each conv block."""
    img_tensor, label = dataset[0]
    class_name = dataset.classes[label]
    activations = []
    hooks = []

    for block in model.features:
        hooks.append(block.block[0].register_forward_hook(
            lambda m, i, o: activations.append(o.detach().cpu())
        ))

    with torch.no_grad():
        model(img_tensor.unsqueeze(0).to(DEVICE))
    for h in hooks:
        h.remove()

    fig, axes = plt.subplots(len(activations), n_maps + 1, figsize=(20, 8))
    denorm = img_tensor.clone()
    for c, (m, s) in enumerate(zip(MEAN, STD)):
        denorm[c] = denorm[c] * s + m
    denorm = denorm.clamp(0, 1).permute(1, 2, 0).numpy()

    for row in range(len(activations)):
        axes[row, 0].imshow(denorm)
        axes[row, 0].set_title(f"Input\n({class_name})", fontsize=8)
        axes[row, 0].axis("off")
        for col in range(n_maps):
            fmap = activations[row][0, col].numpy()
            axes[row, col + 1].imshow(fmap, cmap="viridis")
            axes[row, col + 1].set_title(f"Block {row+1} Map {col}", fontsize=7)
            axes[row, col + 1].axis("off")

    fig.suptitle("Feature Maps per Conv Block", fontsize=14, fontweight="bold")
    plt.tight_layout()
    path = os.path.join(VIZ_DIR, "feature_maps.png")
    plt.savefig(path, dpi=150, bbox_inches="tight")
    plt.show()
    print(f"Saved: {path}")


def show_misclassified(model, loader, dataset, n=16):
    """3d — Misclassified samples."""
    model.eval()
    mis_imgs, mis_labels, mis_preds = [], [], []
    with torch.no_grad():
        for imgs, labels in loader:
            imgs, labels = imgs.to(DEVICE), labels.to(DEVICE)
            _, preds = model(imgs).max(1)
            mask = preds != labels
            mis_imgs.append(imgs[mask].cpu())
            mis_labels.append(labels[mask].cpu())
            mis_preds.append(preds[mask].cpu())
            if sum(len(m) for m in mis_imgs) >= n:
                break

    mis_imgs   = torch.cat(mis_imgs)[:n]
    mis_labels = torch.cat(mis_labels)[:n]
    mis_preds  = torch.cat(mis_preds)[:n]
    classes = dataset.classes

    cols = 4
    rows = (n + cols - 1) // cols
    fig, axes = plt.subplots(rows, cols, figsize=(cols * 3.5, rows * 3.5))
    for i, ax in enumerate(axes.flat):
        if i < len(mis_imgs):
            img = mis_imgs[i].clone()
            for c, (m, s) in enumerate(zip(MEAN, STD)):
                img[c] = img[c] * s + m
            ax.imshow(img.clamp(0, 1).permute(1, 2, 0).numpy())
            ax.set_title(
                f"True: {classes[mis_labels[i]]}\nPred: {classes[mis_preds[i]]}",
                color="red", fontsize=9
            )
        ax.axis("off")
    fig.suptitle("Misclassified Samples", fontsize=14, fontweight="bold")
    plt.tight_layout()
    path = os.path.join(VIZ_DIR, "misclassified.png")
    plt.savefig(path, dpi=150, bbox_inches="tight")
    plt.show()
    print(f"Saved: {path}")


def plot_confusion_matrix(preds, labels, class_names):
    """3e — Confusion matrix for the custom CNN."""
    from sklearn.metrics import confusion_matrix, ConfusionMatrixDisplay
    cm   = confusion_matrix(labels, preds)
    disp = ConfusionMatrixDisplay(confusion_matrix=cm, display_labels=class_names)
    fig, ax = plt.subplots(figsize=(6, 5))
    disp.plot(cmap="Blues", ax=ax)
    ax.set_title("Confusion Matrix", fontsize=14, fontweight="bold")
    plt.tight_layout()
    path = os.path.join(VIZ_DIR, "confusion_matrix.png")
    plt.savefig(path, dpi=150, bbox_inches="tight")
    plt.show()
    print(f"Saved: {path}")

## 5. Transfer learning: EfficientNet-B0

In [ ]:
# BONUS — Fine-tuning EfficientNet-B0 (timm)

def run_efficientnet_bonus(train_loader, val_loader, test_loader, criterion):

    FROZEN_EPOCHS   = 5
    UNFROZEN_EPOCHS = 15
    LR_HEAD  = 1e-3
    LR_FULL  = 1e-4

    pt_model = timm.create_model("efficientnet_b0", pretrained=True, num_classes=2).to(DEVICE)
    print(f"\n=== BONUS: EfficientNet-B0 ({count_params(pt_model):,} params) ===")

    # Stage 1 (train only the new classifier head at a high LR)
    for name, param in pt_model.named_parameters():
        if "classifier" not in name:
            param.requires_grad = False
    opt_pt = optim.Adam(filter(lambda p: p.requires_grad, pt_model.parameters()), lr=LR_HEAD)

    print("[Stage 1] Training head only...")
    for epoch in range(1, FROZEN_EPOCHS + 1):
        train_one_epoch(pt_model, train_loader, criterion, opt_pt)
        _, v_acc, _, _ = evaluate(pt_model, val_loader, criterion)
        print(f"  Epoch [{epoch}/{FROZEN_EPOCHS}] Val Acc: {v_acc*100:.2f}%")

    # Stage 2 (unfreeze everything and fine-tune at a low LR (1e-4) with CosineAnnealingLR for smooth LR decay)
    for param in pt_model.parameters():
        param.requires_grad = True
    opt_pt = optim.Adam(pt_model.parameters(), lr=LR_FULL, weight_decay=1e-4)
    sch_pt = optim.lr_scheduler.CosineAnnealingLR(opt_pt, T_max=UNFROZEN_EPOCHS)

    print("[Stage 2] Fine-tuning all layers...")
    best_pt_acc = 0.0
    pt_path = OUTPUT_DIR / "best_efficientnet.pth"
    for epoch in range(1, UNFROZEN_EPOCHS + 1):
        train_one_epoch(pt_model, train_loader, criterion, opt_pt)
        _, v_acc, _, _ = evaluate(pt_model, val_loader, criterion)
        sch_pt.step()
        if v_acc > best_pt_acc:
            best_pt_acc = v_acc
            torch.save(pt_model.state_dict(), pt_path)
        print(f"  Epoch [{epoch}/{UNFROZEN_EPOCHS}] Val Acc: {v_acc*100:.2f}%")

    pt_model.load_state_dict(torch.load(pt_path, map_location=DEVICE))
    _, pt_test_acc, _, _ = evaluate(pt_model, test_loader, criterion)
    print(f"\nEfficientNet-B0 Test Accuracy: {pt_test_acc*100:.2f}%")
    return pt_test_acc

## 6. Training pipeline

In [ ]:
def main():

    if RUN_OPTUNA_TUNING:
        print(f"\n{'=' * 55}\nRunning Optuna study with {N_TRIALS} trials...\n{'=' * 55}")

        study = optuna.create_study(direction="minimize", study_name="catdog_cnn_hpt")
        study.optimize(objective, n_trials=N_TRIALS)

        tuned = study.best_params

        BEST_HPARAMS_FOUND = {
            "BATCH_SIZE": tuned["batch_size"],
            "NUM_EPOCHS": DEFAULT_HYPERPARAMS["NUM_EPOCHS"],
            "LR": tuned["lr"],
            "WEIGHT_DECAY": tuned["weight_decay"],
            "PATIENCE": tuned["patience"],
        }

        print("\nOptuna study finished.")
        print(f"Best validation loss: {study.best_value:.4f}")
        print(f"Best hyperparameters: {BEST_HPARAMS_FOUND}")
    else:
        BEST_HPARAMS_FOUND = DEFAULT_HYPERPARAMS
        print("\nUsing default hyperparameters for training.")


    # Data loaders should now use the chosen BATCH_SIZE
    train_set, val_set, test_set, train_loader, val_loader, test_loader = \
        get_dataloaders(DATA_DIR, batch_size=BEST_HPARAMS_FOUND["BATCH_SIZE"])
    class_names = train_set.classes

    # Clean up any previous model files before starting new training
    if os.path.exists(MODEL_PATH):
        os.remove(MODEL_PATH)
        print(f"Deleted old model file: {MODEL_PATH}")

    # Custom model initialization
    model = CatDogCNN(num_classes=len(class_names)).to(DEVICE)
    print(f"\nTrainable parameters: {count_params(model):,}")

    # Run training with the chosen hyperparameters
    history = run_training(model, train_loader, val_loader, hyperparameters=BEST_HPARAMS_FOUND)

    criterion = nn.CrossEntropyLoss()
    test_loss, test_acc, test_preds, test_labels = evaluate(model, test_loader, criterion)
    print(f"\nCustom CNN \u2014 Test Loss: {test_loss:.4f} | Test Accuracy: {test_acc*100:.2f}%")

    # Visualisations
    plot_training_curves(history)
    visualize_filters(model)
    visualize_feature_maps(model, val_set)
    show_misclassified(model, test_loader, test_set)
    plot_confusion_matrix(test_preds, test_labels, class_names)

    # Bonus EfficientNet
    pt_test_acc = None
    if RUN_BONUS:
        pt_test_acc = run_efficientnet_bonus(
            train_loader,
            val_loader,
            test_loader,
            criterion,
        )

    # Summary
    print("\n" + "=" * 55)
    print("  RESULTS SUMMARY")
    print("=" * 55)
    print(f"  Custom CatDogCNN   \u2014 Test Acc : {test_acc*100:.2f}%")
    if pt_test_acc is not None:
        print(f"  EfficientNet-B0        \u2014 Test Acc : {pt_test_acc*100:.2f}%")
    print("=" * 55)
    print(f"\nVisualisations saved to: {VIZ_DIR}/")
    print("  \u251c\u2500\u2500 training_curves.png")
    print("  \u251c\u2500\u2500 first_layer_filters.png")
    print("  \u251c\u2500\u2500 feature_maps.png")
    print("  \u251c\u2500\u2500 misclassified.png")
    print("  \u251c\u2500\u2500 confusion_matrix.png")
    print("  \u2514\u2500\u2500 (best_efficientnet.pth if timm is installed)")

## 7. Hyperparameter optimisation with Optuna

In [ ]:
def objective(trial):
    hparams = {
        "BATCH_SIZE": trial.suggest_categorical("batch_size", [16, 32, 64]),
        "NUM_EPOCHS": DEFAULT_HYPERPARAMS["NUM_EPOCHS"],
        "LR": trial.suggest_loguniform("lr", 1e-5, 1e-2),
        "WEIGHT_DECAY": trial.suggest_loguniform("weight_decay", 1e-5, 1e-3),
        "PATIENCE": trial.suggest_int("patience", 3, 7),
    }

    _, _, _, train_loader, val_loader, _ = get_dataloaders(
        DATA_DIR,
        batch_size=hparams["BATCH_SIZE"],
    )

    model = CatDogCNN(num_classes=2).to(DEVICE)

    return train_model_for_optuna(
        model,
        train_loader,
        val_loader,
        trial,
        hparams,
    )

## 8. Run the full experiment

In [ ]:
main()